# Course Project

## Gregory Andriotakis

## 7/3/2025

In this project, we will be using the Default of Credit Card Clients Dataset.

The dataset can be located here: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients


The dataset is designed for a prediction task. Specifically, it looks at if a customer will default on their next credit card payment. We will use MLFlow to train and test three different models, and compare their performance against a baseline. 

In [0]:
# Reading in the data

from pyspark.sql import SparkSession
from pyspark.sql.functions import split, col, when

spark = SparkSession.builder.getOrCreate()

# Define the path to the data
file_path = "dbfs:/FileStore/user/Week8/default_of_credit_card_clients.xls"

# Read in the excel file with com.crealytics.spark.excel
df = spark.read.format("com.crealytics.spark.excel") \ 
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sheetName", "Data") \
    .load(file_path)

# Assign column names
new_columns = [
    "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6",
    "label"
]
df = df.toDF(*new_columns)


The following columns are categorical variables, where a 0 was used to represent missing values. We have to redefine them as we do below in order to handle them correctly during imputation.

In [0]:
from pyspark.sql.functions import col, when

# Treat 0 as missing for EDUCATION and MARRIAGE
categorical_cols_with_zero_missing = ['EDUCATION', 'MARRIAGE']
for c in categorical_cols_with_zero_missing:
    df = df.withColumn(c, when(col(c) == 0, None).otherwise(col(c)))

In [0]:
# Categorical Imputation (Replace with mode)

from pyspark.sql import functions as F

for c in categorical_cols_with_zero_missing:
    mode = df.groupBy(c).count().orderBy(F.desc('count')).first()[0]
    df = df.fillna({c: mode})


In [0]:
# Numerical Imputation (Replace with mean)

from pyspark.ml.feature import Imputer

numeric_cols = [
    "LIMIT_BAL", "AGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]

for c in numeric_cols:
    df = df.withColumn(c, col(c).cast("double"))
    
imputer = Imputer(inputCols=numeric_cols, outputCols=numeric_cols)
df = imputer.fit(df).transform(df)

Below, we define our two new predictors.

Credit Utilization Ratio: This is the proportion of available credit that was used in the prior month. A higher value would indicate higher credit usage, suggesting potential financial stress.

Total Payment Amount Over Last 6 Months: The sum of payments over the last six months. A higher amount may suggest lower risk for default. 



In [0]:
# Feature Engineering

# 1. Credit Utilization Ratio
df = df.withColumn(
    "credit_util_ratio",
    (col("BILL_AMT1") / col("LIMIT_BAL")).cast("double")
)

# 2. Total Payment Amount Over Last 6 Months
df = df.withColumn(
    "total_pay_amt_6m",
    sum([col(f"PAY_AMT{i}") for i in range(1, 7)])
)


In [0]:
# Only keep rows where label is 0 or 1
df = df.filter(col('label').isin([0, 1]))
df = df.withColumn('label', col('label').cast('int'))

# Prepare feature list
categorical_cols = ['SEX', 'EDUCATION', 'MARRIAGE'] + ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
feature_cols = numeric_cols + ["credit_util_ratio", "total_pay_amt_6m"] + categorical_cols
feature_cols = [c for c in feature_cols if c != "label"]  # Remove label if present

In [0]:
# Check label distribution
df.groupBy('label').count().show()

+-----+-----+
|label|count|
+-----+-----+
|    1| 6636|
|    0|23364|
+-----+-----+



In [0]:
# Stratified split

pdf = df.toPandas()

from sklearn.model_selection import train_test_split

# Ensure no missing values remain
pdf = pdf.fillna(pdf.mean(numeric_only=True))
pdf = pdf.fillna(0)

# Split into train, test, and validation stratified by label
train_pd, temp_pd = train_test_split(pdf, test_size=0.2, stratify=pdf['label'], random_state=42)
test_pd, val_pd = train_test_split(temp_pd, test_size=0.5, stratify=temp_pd['label'], random_state=42)

train_df = spark.createDataFrame(train_pd)
test_df = spark.createDataFrame(test_pd)
val_df = spark.createDataFrame(val_pd)

Here, we create a baseline model as a reference for our models' performances. 

In [0]:
# Baseline Model

from pyspark.sql.functions import lit

# Identify the majority class
majority_class = train_df.groupBy("label").count().orderBy(col("count").desc()).first()[0]
print(f"Majority class (baseline prediction): {majority_class}")

# Create baseline values using majority class
test_pred_baseline = test_df.withColumn("baseline_prediction", lit(float(majority_class)))
val_pred_baseline = val_df.withColumn("baseline_prediction", lit(float(majority_class)))

Majority class (baseline prediction): 0


In [0]:
# Evaluating Baseline

from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Define evaluators (AUC and Accuracy)
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="baseline_prediction", metricName="areaUnderROC")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="baseline_prediction", metricName="accuracy")

# Generate test and validation predictions
baseline_test_auc = evaluator_auc.evaluate(test_pred_baseline)
baseline_val_auc = evaluator_auc.evaluate(val_pred_baseline)
baseline_test_acc = evaluator_acc.evaluate(test_pred_baseline)
baseline_val_acc = evaluator_acc.evaluate(val_pred_baseline)

# Display results
print(f"Baseline Test AUC: {baseline_test_auc:.3f}, Validation AUC: {baseline_val_auc:.3f}")
print(f"Baseline Test Accuracy: {baseline_test_acc:.3f}, Validation Accuracy: {baseline_val_acc:.3f}")

Baseline Test AUC: 0.500, Validation AUC: 0.500
Baseline Test Accuracy: 0.779, Validation Accuracy: 0.779


From the baseline's AUC of 0.5, we can tell that the model does no better than randomly guessing if a customer will default or not. 

We see that the baseline has an accuracy of 77.9%. The accuracy being this high is potentially due to the target class imbalance that was demonstrated earlier. 

In [0]:
# Encoding, Scaling, and Assembling

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, MinMaxScaler

# Convert string values into numeric indicies
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]

# Apply One-Hot Encoding to categorical variables
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in categorical_cols]

# Asseble numerical features into a vector
assembler_num = VectorAssembler(inputCols=[c for c in feature_cols if c not in categorical_cols], outputCol="num_features")

# Apply Min-Max Scaler to numerical features
scaler = MinMaxScaler(inputCol="num_features", outputCol="num_features_scaled")

# Assemble all features into a vector
assembler_all = VectorAssembler(
    inputCols=[f"{c}_ohe" for c in categorical_cols] + ["num_features_scaled"],
    outputCol="features"
)



Below, we define, train, and test our three models.

The three model types include:
1. Logistic Regression
2. Random Forest
3. Decision Tree

Like the baseline, we will assess models based on AUC and accuracy.

In [0]:
# Model construction and MLFlow 

from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, DecisionTreeClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import mlflow
import mlflow.spark
from pyspark.mllib.evaluation import MulticlassMetrics

# Define models
models = {
    "LogisticRegression": LogisticRegression(featuresCol="features", labelCol="label"),
    "RandomForest": RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50),
    "DecisionTree": DecisionTreeClassifier(featuresCol="features", labelCol="label"),
}

# Define evaluators (AUC and Accuracy)
evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# Define code to generate confusion matrix
def print_confusion_matrix(pred_df):
    preds_and_labels = pred_df.select("prediction", "label") \
        .rdd.map(lambda row: (float(row["prediction"]), float(row["label"])))
    metrics = MulticlassMetrics(preds_and_labels)
    print("Confusion Matrix:")
    print(metrics.confusionMatrix().toArray())

# Train, log, and evaluate models using MLFlow
for name, clf in models.items():
    with mlflow.start_run(run_name=name):
        pipeline = Pipeline(stages=indexers + encoders + [assembler_num, scaler, assembler_all, clf])
        model = pipeline.fit(train_df)
        pred = model.transform(test_df)
        val_pred = model.transform(val_df)
        auc = evaluator.evaluate(pred)
        val_auc = evaluator.evaluate(val_pred)
        acc = evaluator_acc.evaluate(pred)
        val_acc = evaluator_acc.evaluate(val_pred)
        mlflow.log_param("model", name)
        mlflow.log_metric("test_auc", auc)
        mlflow.log_metric("val_auc", val_auc)
        mlflow.log_metric("test_acc", acc)
        mlflow.log_metric("val_acc", val_acc)
        mlflow.spark.log_model(model, name)
        print(f"{name} Test AUC: {auc:.3f}, Validation AUC: {val_auc:.3f}")
        print(f"{name} Test Accuracy: {acc:.3f}, Validation Accuracy: {val_acc:.3f}")
        print_confusion_matrix(pred)

2025/06/27 18:57:21 INFO mlflow.spark: Inferring pip requirements by reloading the logged model from the databricks artifact repository, which can be time-consuming. To speed up, explicitly specify the conda_env or pip_requirements when calling log_model().


2025/06/27 18:58:38 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: dbfs:/databricks/mlflow-tracking/1191042229762790/3a5d1e0176684adab79fdcd3becf4bee/artifacts/LogisticRegression/sparkml, flavor: spark). Fall back to return ['pyspark==3.5.2']. Set logging level to DEBUG to see the full traceback. 


Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

LogisticRegression Test AUC: 0.772, Validation AUC: 0.741
LogisticRegression Test Accuracy: 0.821, Validation Accuracy: 0.811


/databricks/spark/python/pyspark/sql/context.py:165: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


...

RandomForest Test AUC: 0.782, Validation AUC: 0.743
RandomForest Test Accuracy: 0.811, Validation Accuracy: 0.801
Confusion Matrix:
[[2271.   65.]
 [ 502.  162.]]


2025/06/27 19:04:02 INFO mlflow.spark: Inferring pip requirements by reloading the logged model from the databricks artifact repository, which can be time-consuming. To speed up, explicitly specify the conda_env or pip_requirements when calling log_model().


2025/06/27 19:05:16 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: dbfs:/databricks/mlflow-tracking/1191042229762790/f3f7bbfd899e48548a4a47ee64f509c9/artifacts/DecisionTree/sparkml, flavor: spark). Fall back to return ['pyspark==3.5.2']. Set logging level to DEBUG to see the full traceback. 


Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

DecisionTree Test AUC: 0.261, Validation AUC: 0.299
DecisionTree Test Accuracy: 0.819, Validation Accuracy: 0.811
Confusion Matrix:
[[2240.   96.]
 [ 448.  216.]]


## Model Analysis

The best performing model was the Logistic Regression model.

The Logistic Regression model obtained higher AUC scores and accuracy scores than the Decision Tree model. While the AUC was slightly higher in the Random Forest model than the Logistic Regression model, it suffered in accuracy compared to the Logistic Regression. 

The Logistic Regression model obtained a test AUC of 0.772, which is much higher than the baseline model's AUC of 0.5. This tells us that compared to randomly guessing, the Logistic Regression model is more capable at actually identifying each target class. 

The Logistic Regression's test accuracy was 82.1%, which is an improvement over the baseline's accuracy of 77.9%. The model obtained a validation accuracy of 81.1%, which is lower than the test accuracy, but not by a great amount. This suggests that the Logistic Regression model is fairly capable at generalizing towards new observations. 

Overall, it is clear that the Logistic Regression model is an improvement over the baseline and demonstrates predictive power beyond naive guessing. 